# Crop Guard - 农作物害虫检测模型训练

基于 YOLO11n 的 30 类农作物害虫检测模型微调训练。

**使用前请先切换到 GPU 运行时：** 运行时 → 更改运行时类型 → T4 GPU

## 1. 环境准备

In [ ]:
# 检查 GPU
!nvidia-smi

In [ ]:
# 安装 ultralytics
!pip install ultralytics -q

## 2. 上传数据集

将本地 `database/filtered_dataset/` 文件夹压缩为 zip 后上传。

压缩命令（本地终端执行）：
```bash
cd crop-guard-platform/database
zip -r filtered_dataset.zip filtered_dataset/
```

In [ ]:
# 上传 filtered_dataset.zip
from google.colab import files
uploaded = files.upload()

In [ ]:
# 解压数据集
!unzip -q filtered_dataset.zip -d /content/
print("解压完成")
!ls /content/filtered_dataset/images/train/ | wc -l
!ls /content/filtered_dataset/images/val/ | wc -l

## 3. 创建 data.yaml

In [ ]:
import yaml

data = {
    'path': '/content/filtered_dataset',
    'train': 'images/train',
    'val': 'images/val',
    'nc': 30,
    'names': [
        'rice_leaf_roller', 'rice_leaf_caterpillar', 'paddy_stem_maggot',
        'asiatic_rice_borer', 'yellow_rice_borer', 'rice_gall_midge',
        'Rice_Stemfly', 'brown_plant_hopper', 'white_backed_plant_hopper',
        'small_brown_plant_hopper', 'rice_water_weevil', 'rice_leafhopper',
        'corn_borer', 'army_worm', 'aphids',
        'english_grain_aphid', 'black_cutworm', 'large_cutworm',
        'yellow_cutworm', 'red_spider', 'mole_cricket',
        'grub', 'wireworm', 'Prodenia_litura',
        'beet_army_worm', 'cabbage_army_worm', 'flea_beetle',
        'Locustoidea', 'blister_beetle', 'wheat_sawfly'
    ]
}

with open('/content/data.yaml', 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print('data.yaml 已创建')
!cat /content/data.yaml

## 4. 开始训练

In [ ]:
from ultralytics import YOLO

# 加载预训练模型
model = YOLO('yolo11n.pt')

# 开始训练
results = model.train(
    data='/content/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project='/content/runs',
    name='crop_guard_v1',
    patience=20,
    save=True,
    verbose=True
)

## 5. 评估模型

In [ ]:
# 在验证集上评估
metrics = model.val()
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

## 6. 导出模型

In [ ]:
# 下载训练好的模型
from google.colab import files
files.download('/content/runs/crop_guard_v1/weights/best.pt')
print('模型已下载！将 best.pt 放到 crop-guard-platform/backend/ 目录下即可使用')

## 7. 测试推理（可选）

In [ ]:
# 用训练好的模型测试一张图片
from ultralytics import YOLO
import glob

best_model = YOLO('/content/runs/crop_guard_v1/weights/best.pt')

# 从验证集随机选一张测试
test_images = glob.glob('/content/filtered_dataset/images/val/*.jpg')
if test_images:
    results = best_model.predict(test_images[0], save=True, conf=0.25)
    print(f'检测到 {len(results[0].boxes)} 个目标')
    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        print(f'  - {data["names"][cls_id]}: {conf:.2%}')
    from IPython.display import Image
    Image('/content/runs/predict/*.jpg')
else:
    print('未找到测试图片')